# German power market in 2024: what the ENTSO-E data shows

First look at real data for the DE-LU bidding zone and its neighbours, before building the model.
This is the input and validation data for study block 1 (does the model reproduce 2024?).

Data: ENTSO-E Transparency Platform, downloaded with `entsoe-py` and cached in `data/raw/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
import entsoe_data as ed

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True

YEAR = 2024
client = ed.get_client()

## 1. Download data

The first run takes a few minutes. Later runs read the CSV files from `data/raw/`.

In [ ]:
prices_de = ed.day_ahead_prices(client, "DE_LU", YEAR)
load_de = ed.load(client, "DE_LU", YEAR)
gen_de = ed.generation(client, "DE_LU", YEAR)

print(f"Prices:     {len(prices_de)} hours")
print(f"Load:       {len(load_de)} hours")
print(f"Generation: {gen_de.shape[0]} hours, {gen_de.shape[1]} technologies")

In [ ]:
# Quick data check: missing hours and gaps
for name, s in {"prices": prices_de, "load": load_de}.items():
    print(f"{name}: {s.isna().sum()} missing hours")
print("Generation columns:", list(gen_de.columns))

## 2. Prices: key figures

In [ ]:
summary = pd.Series({
    "Average price (EUR/MWh)": prices_de.mean(),
    "Median price (EUR/MWh)": prices_de.median(),
    "Min price (EUR/MWh)": prices_de.min(),
    "Max price (EUR/MWh)": prices_de.max(),
    "Hours with negative prices": (prices_de < 0).sum(),
    "Hours above 150 EUR/MWh": (prices_de > 150).sum(),
}).round(1)
summary

## 3. Monthly average price

In [ ]:
monthly = prices_de.resample("MS").mean()
ax = monthly.plot(kind="bar", color="tab:blue")
ax.set_xticklabels([d.strftime("%b") for d in monthly.index], rotation=0)
ax.set_ylabel("EUR/MWh")
ax.set_xlabel("")
ax.set_title(f"DE-LU day-ahead price, monthly average {YEAR}")
plt.tight_layout()

## 4. Price duration curve

All hours of the year sorted from the highest to the lowest price.
The tail on the right shows the hours with negative prices.

In [ ]:
pdc = prices_de.sort_values(ascending=False).reset_index(drop=True)
ax = pdc.plot()
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Hours of the year")
ax.set_ylabel("EUR/MWh")
ax.set_title(f"DE-LU price duration curve {YEAR}")
plt.tight_layout()

## 5. Average price by hour of day and month

The midday dip in spring and summer is the solar effect.

In [ ]:
pivot = prices_de.groupby([prices_de.index.month, prices_de.index.hour]).mean().unstack(0)
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(pivot, aspect="auto", cmap="RdYlBu_r", origin="lower")
ax.set_xticks(range(12))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_ylabel("Hour of day")
ax.set_title(f"DE-LU average price by hour and month {YEAR} (EUR/MWh)")
fig.colorbar(im, ax=ax, label="EUR/MWh")
plt.tight_layout()

## 6. Generation mix by month

In [ ]:
monthly_gen = gen_de.resample("MS").sum() / 1e6  # MWh -> TWh (hourly MW values)
# keep the largest technologies, group the rest
top = monthly_gen.sum().sort_values(ascending=False).index[:8]
mix = monthly_gen[top].copy()
mix["Other"] = monthly_gen.drop(columns=top).sum(axis=1)

ax = mix.plot(kind="bar", stacked=True, figsize=(11, 5), width=0.85)
ax.set_xticklabels([d.strftime("%b") for d in mix.index], rotation=0)
ax.set_ylabel("TWh")
ax.set_xlabel("")
ax.set_title(f"DE-LU generation by technology {YEAR}")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()

In [ ]:
annual_mix = (gen_de.sum() / 1e6).sort_values(ascending=False).round(1)
annual_mix.to_frame("TWh")

## 7. Prices in neighbouring zones

Monthly averages for the zones of the planned model.

In [ ]:
zones = ["DE_LU", "FR", "NL", "BE", "PL", "CZ", "AT", "DK_1"]
prices = {}
for z in zones:
    try:
        prices[z] = ed.day_ahead_prices(client, z, YEAR)
    except Exception as e:
        print(f"{z}: no data ({type(e).__name__})")
prices = pd.DataFrame(prices)

ax = prices.resample("MS").mean().plot(marker="o")
ax.set_ylabel("EUR/MWh")
ax.set_xlabel("")
ax.set_title(f"Monthly average day-ahead prices {YEAR}")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()

In [ ]:
prices.mean().sort_values().round(1).to_frame("Average price 2024 (EUR/MWh)")

## 8. Notes for the model

Write down here what the data shows, for example:

- average level and spread of prices,
- how many negative hours and when they occur,
- which zones are cheaper or more expensive than DE-LU,
- which technologies set the price in which season.

These points become the checks for study block 1.